# **Spam Classifier (Logistic Regression from Scratch)**
Binary classification model for detecting spam messages using:

- Logistic regression
- Cross-entropy loss
- Gradient descent
- Evaluation metrics (precision, recall)

In [52]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import re

In [53]:
#load the dataset
df = pd.read_csv("spam.csv", encoding='latin-1')[['v1','v2']]
df

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will Ì_ b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [54]:
df.columns = ['label', 'message']
df['label'] = df['label'].str.strip().str.lower()
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
df

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


In [55]:
spam = df[df['label'] == 1]
ham = df[df['label'] == 0] #creates 2 datasets

ham_sampled = ham.sample(len(spam), random_state=42)

df = pd.concat([spam, ham_sampled]) #combine the datasets
df = df.sample(frac=1, random_state=42).reset_index(drop=True) #shuffle data set

In [56]:
print(df.shape)
print(df['label'].value_counts())

(1494, 2)
label
0    747
1    747
Name: count, dtype: int64


In [57]:
#data cleaing i.e. texts
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]','', text)
    return text

df['cleaned_msg'] = df['message'].apply(clean_text)
df

,label,message,cleaned_msg
0,0,"Come to mu, we're sorting out our narcotics si...",come to mu were sorting out our narcotics situ...
1,0,We can go 4 e normal pilates after our intro...,we can go 4 e normal pilates after our intro
2,0,Have you laid your airtel line to rest?,have you laid your airtel line to rest
3,0,Gokila is talking with you aha:),gokila is talking with you aha
4,0,We confirm eating at esplanade?,we confirm eating at esplanade
...,...,...,...
1489,0,"I'm in a meeting, call me later at",im in a meeting call me later at
1490,0,S da..al r above &lt;#&gt;,s daal r above ltgt
1491,0,"Love isn't a decision, it's a feeling. If we c...",love isnt a decision its a feeling if we could...
1492,0,Wat makes some people dearer is not just de ha...,wat makes some people dearer is not just de ha...


In [70]:
#limited vocab
all_words= []
for text in df['cleaned_msg']:
    all_words.extend(text.split())

common_words = Counter(all_words).most_common(1000)

In [71]:
vocab = {word: i for i, (word, _) in enumerate(common_words)}

In [72]:
#vectorization and normalization
def vectorize(text):
    vec = np.zeros(len(vocab))
    for word in text.split():
        if word in vocab:
            vec[vocab[word]] += 1
    return vec

X = []
for text in df['cleaned_msg']:
    X.append(vectorize(text))
X = np.array(X)
Y = df['label'].values

In [61]:
#training and testing split
idx = np.random.permutation(len(X))

X = X[idx]
Y = Y[idx]

split = int(0.7 * len(X))

x_train, x_test = X[:split], X[split:]
y_train, y_test = Y[:split], Y[split:]

## Logistic Regression Model
- Sigmoid for probability estimation
- Cross-entropy loss for optimization
- Gradient descent for parameter updates

In [62]:
#sigmoid and loss functions
def sigmoid(z):
    """Compute the sigmoid function"""
    return 1/(1 + np.exp(-z))

def cross_entropy_loss(y_true, p_pred):
    eps = 1e-15
    p_pred = np.clip(p_pred, eps, 1 - eps)
    return -np.mean(y_true*np.log(p_pred) + (1-y_true)*np.log(1-p_pred))

In [63]:
w=np.zeros(x_train.shape[1])
b=0
lr=0.1
epochs=350

In [64]:
for i in range(epochs):
    Z = x_train @ w + b
    p=sigmoid(Z)
    dw=(1/len(x_train)) * (x_train.T@(p-y_train))
    db=(1/len(x_train)) * np.sum(p-y_train)

    w-= lr*dw
    b-=lr*db

## Evaluation Metrics

The model performance is evaluated using the following metrics:
- Confusion Matrix
- Accuracy
- Precision
- Recall
  

In [65]:
#predicitons
Z = x_test @ w + b
p = sigmoid(Z)

threshold = 0.5
y_pred = (p >= threshold).astype(int)

In [66]:
#metrics
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))

In [67]:
precision = TP / (TP + FP) if (TP + FP) != 0 else 0
recall = TP / (TP + FN) if (TP + FN) != 0 else 0
accuracy = (TP + TN) / (TP + TN + FP + FN)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

Accuracy: 0.9220489977728286
Precision: 0.9393939393939394
Recall: 0.8899521531100478


In [68]:
def predict_spam(text):
    text = clean_text(text)

    vec = np.zeros(len(vocab))
    for word in text.split():
        if word in vocab:
            vec[vocab[word]] += 1
    z = np.dot(vec, w) + b
    p = 1 / (1 + np.exp(-z))

    print("DEBUG → z:", z, "p:", p)  

    return "Most Likely Spam" if p >= 0.4 else "Most Likely Ham"

In [73]:
print(predict_spam("Congratulations! You won a free prize"))
print(predict_spam("You have a meeting at 10pm"))

DEBUG → z: 0.2733588430077034 p: 0.5679173094686406
Most Likely Spam
DEBUG → z: -0.9811615742630805 p: 0.2726613627469528
Most Likely Ham
